In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import label_binarize

random.seed(11924906)

# Logistics
df = pd.read_csv('musicData.csv')
df = df.drop(columns=['instance_id', 'artist_name', 'track_name', 'obtained_date']) # Unnecessary columns

# Convert numeric data and categorical data
numeric_columns = ['popularity', 'acousticness', 'danceability', 'duration_ms', 'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo', 'valence']
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df['key'] = df['key'].astype(str)
df['mode'] = df['mode'].astype(str)
df['music_genre'] = df['music_genre'].astype(str)

# Define features (numerical and categorical)
num_features = ['popularity', 'acousticness', 'danceability', 'duration_ms', 'energy','instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo', 'valence']
cat_features = ['key', 'mode']

# Pipelines to help preprocess the data
num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='mean')),('scaler', StandardScaler())])
cat_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num', num_pipeline, num_features),('cat', cat_pipeline, cat_features)]) # Combine the steps

# Encode the target labels
label_encoder = LabelEncoder()
df['genre_label'] = label_encoder.fit_transform(df['music_genre'])

# Split data by genre
X_train_list, X_test_list = [], []
for genre in df['music_genre'].unique():
    genre_df = df[df['music_genre'] == genre]
    test_size = min(500, len(genre_df)) # If there is not 500, then use all that is available just in case
    test_samples = genre_df.sample(n=test_size, random_state=11924906) # Use the random state from my NetID
    train_samples = genre_df.drop(test_samples.index)
    X_train_list.append(train_samples)
    X_test_list.append(test_samples)

# Combine the training and test samples
train_df = pd.concat(X_train_list)
test_df = pd.concat(X_test_list)

X_train = train_df[num_features + cat_features]
y_train = label_encoder.transform(train_df['music_genre'])
X_test = test_df[num_features + cat_features]
y_test = label_encoder.transform(test_df['music_genre'])

# Preprocess the data
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# Perform PCA for dimensionality reduction
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)

# Plot PCA 
plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_train_pca[:, 0], y=X_train_pca[:, 1], hue=train_df['music_genre'], palette='tab10', s=7.5)
plt.title("PCA of Spotify Songs (Colored by Genre)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Train a Random Forest classifier
clf = OneVsRestClassifier(RandomForestClassifier(n_estimators=100, random_state=11924906))
clf.fit(X_train, y_train)

y_scores = clf.predict_proba(X_test)
classes = np.unique(y_train)

# Binarize the labels
y_test_binary = label_binarize(y_test, classes=classes)

# Calculate and print the AUC score
final_auc = roc_auc_score(y_test_binary, y_scores, average='macro')
print(f"Final AUC: {final_auc:.4f}")

# Plot ROC curve per genre
plt.figure(figsize=(8, 6))
for i in range(len(classes)):
    fpr, tpr, _ = roc_curve(y_test_binary[:, i], y_scores[:, i])
    roc_auc = auc(fpr, tpr)
    genre_name = label_encoder.inverse_transform([classes[i]])[0]
    plt.plot(fpr, tpr, lw=2, label=f"{genre_name} (AUC = {roc_auc:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Each Genre')
plt.legend(loc='lower right', fontsize='small')
plt.grid(True)
plt.tight_layout()
plt.show()

# Extra Analysis

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_train_pca[:, 0], y=X_train_pca[:, 1], hue=train_df['music_genre'], palette='tab10', alpha=0.6, s=7.5)

# Add contributing vectors
loadings = pca.components_.T
for i, feature in enumerate(num_features):
    plt.arrow(0, 0, loadings[i, 0]*5, loadings[i, 1]*5, color='black', alpha=0.6, head_width=0.1)
    plt.text(loadings[i, 0]*5.2, loadings[i, 1]*5.2, feature, color='black', ha='center', va='center', fontsize=8)

plt.title("PCA Biplot: Spotify Genres with Feature Vectors")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Genre', fontsize=8)
plt.tight_layout()
plt.show()